# 教程 02：材料性质预测

**学习目标** ⭐
- 了解晶体图神经网络的基本概念
- 学习使用 DAO 基础算子库
- 掌握材料描述符计算和性质预测的流程

---

## 背景

材料性质预测是计算材料科学的核心任务之一，目标是基于原子结构快速预测材料的电子、力学、热学等性质，避免昂贵的实验试错。

### 核心方法

- **晶体图神经网络 (CGNN)**：将晶体结构编码为图（原子→节点，键→边），用消息传递机制学习结构-性质关系
- **材料描述符**：从原子坐标和晶格参数中提取的数值特征，如径向分布函数、角度分布、键长分布等
- **相图预测**：基于热力学数据预测不同温度/压力下的稳定相

本仓库的 DAO 库提供了材料科学中的基础算子：图构建、周期性边界条件、扩散算子等。

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(repo_root, 'simulation', 'MaterialPropertyPrediction'))

try:
    from DAO import (
        build_crystal_graph,
        pbc_pairwise_distance,
        basis_ops,
        attention_ops,
    )
    print("DAO 库加载成功 ✅")
except ImportError as e:
    print(f"DAO 库加载失败: {e}")
    print("需要先安装 PyTorch: pip install torch")

## 示例：构建晶体图

以金刚石结构的硅晶体为例，构建其图表示。

In [ ]:
import torch
import numpy as np

# 金刚石结构元胞参数
lattice = np.array([
    [0.0, 0.5, 0.5],
    [0.5, 0.0, 0.5],
    [0.5, 0.5, 0.0],
]) * 5.43  # 硅的晶格常数 (Å)

fractional_coords = np.array([
    [0.0, 0.0, 0.0],
    [0.25, 0.25, 0.25],
], dtype=np.float32)

atom_types = np.array([14, 14], dtype=np.int32)  # 硅的原子序数

print(f"晶格矩阵:\n{lattice}")
print(f"原子分数坐标:\n{fractional_coords}")
print(f"原子类型: {atom_types}")

In [ ]:
# 计算径向分布函数 (RDF) 作为材料描述符
# 使用原子间距离的直方图来表征结构

def compute_rdf(positions, box_size, bins=50, rmax=5.0):
    n_atoms = len(positions)
    distances = []
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            r = positions[i] - positions[j]
            r = r - np.round(r / box_size) * box_size
            d = np.linalg.norm(r)
            if d < rmax:
                distances.append(d)
    hist, edges = np.histogram(distances, bins=bins, range=(0, rmax))
    bin_centers = (edges[:-1] + edges[1:]) / 2
    return bin_centers, hist.astype(np.float32)

# 构建 2x2x2 超胞
supercell = []
for i in range(2):
    for j in range(2):
        for k in range(2):
            offset = np.array([i, j, k], dtype=np.float32)
            for atom in fractional_coords:
                cart = (atom + offset) @ lattice
                supercell.append(cart)

supercell = np.array(supercell)
box_size = 2 * np.array([5.43, 5.43, 5.43])

bin_centers, rdf = compute_rdf(supercell, box_size)
print(f"RDF 过 {len(bin_centers)} 个 bin，非零计数 {np.sum(rdf > 0)}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.bar(bin_centers, rdf, width=bin_centers[1] - bin_centers[0], alpha=0.7)
    plt.xlabel("r (Å)")
    plt.ylabel("成键计数")
    plt.title("硅晶体径向分布函数")
    plt.grid(alpha=0.3)
    plt.show()
except ImportError:
    print("需要 matplotlib: pip install matplotlib")

## 总结 📋

- 材料性质预测将晶体结构编码为机器可学习的特征
- RDF 是最基础的结构描述符，反映原子间的空间分布
- DAO 库提供了材料科学中的基础算子
- 更复杂的 CGNN 模型可在此基础描述符上进行端到端学习

## 思考练习 ✏️

1. 尝试对 NaCl 结构（岩盐型）计算 RDF，观察与金刚石结构的差异
2. 修改截断半径 rmax，观察 RDF 的变化
3. 思考：如何将 RDF 描述符作为 CGNN 的输入特征？